# SpaceX interactive dashboard — IBM 56-launch dataset

Reproducible source cells implementing the required Plotly Dash site selector, successful landing pie chart, and payload-filtered scatter. Run with `pip install dash pandas plotly` and `spacex_launch_dash.csv` in the same directory.

In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df=pd.read_csv('spacex_launch_dash.csv')
assert {'Launch Site','class','Payload Mass (kg)','Booster Version Category'}.issubset(df.columns)
print(df.groupby('Launch Site')['class'].agg(['count','sum','mean']))

In [ ]:
app=Dash(__name__)
sites=['ALL']+sorted(df['Launch Site'].unique().tolist())
app.layout=html.Div([html.H1('SpaceX Launch Records Dashboard'),
 html.Label('Launch site'),dcc.Dropdown(sites,'ALL',id='site'),
 dcc.Graph(id='pie'),html.Label('Payload mass (kg)'),
 dcc.RangeSlider(0,10000,step=500,value=[0,10000],id='mass'),dcc.Graph(id='scatter')])
@app.callback(Output('pie','figure'),Input('site','value'))
def pie(site):
 if site=='ALL':
  s=df[df['class']==1].groupby('Launch Site').size().reset_index(name='Successful Landings')
  return px.pie(s,names='Launch Site',values='Successful Landings',title='Successful landings by site')
 sub=df[df['Launch Site']==site]
 counts=sub['class'].map({0:'Not successful',1:'Successful'}).value_counts().reset_index()
 counts.columns=['Landing','Count']
 return px.pie(counts,names='Landing',values='Count',title='Landing outcome: '+site)
@app.callback(Output('scatter','figure'),Input('site','value'),Input('mass','value'))
def scatter(site,rg):
 sub=df[(df['Payload Mass (kg)']>=rg[0])&(df['Payload Mass (kg)']<=rg[1])]
 if site!='ALL':sub=sub[sub['Launch Site']==site]
 return px.scatter(sub,x='Payload Mass (kg)',y='class',color='Booster Version Category',hover_data=['Launch Site'],title='Payload mass versus first-stage landing class')
if __name__=='__main__':app.run(debug=False,port=8050)

## Verified data summary

The uploaded 56-row IBM dataset contains 24 class=1 landings. Site-level results: KSC LC-39A 10/13, CCAFS LC-40 7/26, VAFB SLC-4E 4/10, and CCAFS SLC-40 3/7. This notebook provides functional code but the cells have not been saved with execution outputs; run locally to capture live application screenshots.